This notebook sets up the schemas and volumes for the ETL pipeline, as described in https://github.com/RiccardoLui/Data_Lakehouse_Databricks.

A medallion architecture is created, with csv datasets imported into a volume under the Bronze schema.

In [0]:
-- Create catalog
CREATE CATALOG IF NOT EXISTS data_lakehouse_databricks;
USE CATALOG data_lakehouse_databricks;

-- Create schema
CREATE SCHEMA IF NOT EXISTS data_lakehouse_databricks.bronze;
CREATE SCHEMA IF NOT EXISTS data_lakehouse_databricks.silver;
CREATE SCHEMA IF NOT EXISTS data_lakehouse_databricks.gold;
--USE SCHEMA bronze;

-- Create the ingestion volume
CREATE VOLUME IF NOT EXISTS data_lakehouse_databricks.bronze.bronze_vol

In [0]:
%python
import os

# Source directory in workspace
source_base_dir = '/Workspace/Users/2018luir@gmail.com/Data_Lakehouse_Databricks/datasets/engineering/'

# Target volume path
volume_base_path = '/Volumes/Data_Lakehouse_Databricks/Bronze/Bronze_Vol/'

# Folders to process
folders = ['source_erp', 'source_crm']

# Copy files from workspace to volume
for folder in folders:
    source_folder = source_base_dir + folder + '/'
    target_folder = volume_base_path + folder + '/'
    
    print(f"\nProcessing folder: {folder}")
    print(f"Source: {source_folder}")
    print(f"Target: {target_folder}")
    
    # Create target folder in volume if it doesn't exist
    dbutils.fs.mkdirs(target_folder)
    
    # List all CSV files in source folder
    try:
        files = dbutils.fs.ls(source_folder)
        csv_files = [f for f in files if f.name.endswith('.csv')]
        
        print(f"Found {len(csv_files)} CSV files")
        
        # Copy each CSV file
        for file_info in csv_files:
            source_path = file_info.path
            target_path = target_folder + file_info.name
            
            print(f"  Copying {file_info.name}...", end=" ")
            dbutils.fs.cp(source_path, target_path, recurse=False)
            print("✓")
            
    except Exception as e:
        print(f"  ✗ Error processing {folder}: {str(e)}")

print("\n=== Copy operation completed ===")
print(f"\nFiles are now in: {volume_base_path}")